In [ ]:
conda create -n myenv python=3.10 -y

In [1]:
!pip install torch
!pip install pandas
!pip install numpy 

In [1]:
import torch 
import pandas as pd
import numpy as np

In [2]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [3]:
dataset["train"].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

Model time

In [2]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 26.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 23.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]


In [1]:
!pip install tqdm


In [30]:
import torch
import torch.nn as nn
from transformers import BertConfig
from tqdm.auto import tqdm

class CustomFeedForwardLayer(nn.Module):
    def __init__(self,config, rank = 64):
        super().__init__()

        self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)

        self.A = nn.Parameter(
            torch.randn(config.intermediate_size, config.intermediate_size, rank)
        )
        self.A2 = nn.Parameter(
            torch.randn(config.hidden_size, config.hidden_size, rank)
        )

    def forward(self, x):

        x = self.linear1(x)
        Ax = torch.einsum("bsi,oik->bsok", x, self.A)
        quad = torch.sum(Ax * Ax, dim=-1)
        x = self.activation(x+quad)

        x = self.linear2(x)
        Ax = torch.einsum("bsi,oik->bsok", x, self.A2)
        quad = torch.sum(Ax * Ax, dim=-1)
        return x + quad
    
class BertLayer(nn.Module):
    def __init__(self,config):
        
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=config.hidden_size,
            num_heads=config.num_attention_heads,
            batch_first=True)
        
        self.cff = CustomFeedForwardLayer(config)

        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)

    def forward(self, x,attention_mask=None):

        attention_output,_ = self.attention(x,x,x)
        x = self.norm1(x + attention_output)

        cffn_output = self.cff(x)
        x = self.norm2(x + cffn_output)
 
        return x
        
class BertEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.hidden_size
        )

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )

        self.layer_norm = nn.LayerNorm(config.hidden_size)

    def forward(self, input_ids):

        seq_length = input_ids.size(1)

        position_ids = torch.arange(
            seq_length, device=input_ids.device
        ).unsqueeze(0)

        word_embeddings = self.word_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        embeddings = word_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)

        return embeddings
    

class BertEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.layers = nn.ModuleList([
            BertLayer(config) for _ in range(config.num_hidden_layers)
        ])

    def forward(self, x, attention_mask=None):
        for layer in self.layers:
            x = layer(x, attention_mask)
        return x


class MyBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)

    def forward(self, input_ids, attention_mask=None,embed = False):

        x = self.embeddings(input_ids)
        
        if embed :
            return x 
        
        x = self.encoder(x, attention_mask)
        logits = self.lm_head(x)


        return logits

        # return x


In [31]:
config = BertConfig(
    vocab_size=30522,
    hidden_size=256,
    num_hidden_layers=12,
    num_attention_heads=4,
    intermediate_size=256,
    output_dim = 256
)

model = MyBertModel(config)

In [71]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 2. Loss Function (CrossEntropy for Masked Language Modeling or Classification)
criterion = nn.CrossEntropyLoss().to(device)

# 3. Optimizer (Weight decay is crucial for Transformers)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

In [17]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [18]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [19]:
tokenized_datasets["train"]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [43]:
raw_input_ids = tokenized_datasets["test"][0]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor = torch.tensor(raw_input_ids).unsqueeze(0).to(device)

# 3. Now run the model
output = model(input_tensor)
print(output.shape)

torch.Size([1, 128, 30522])


In [35]:
a = model(input_tensor,None,True)

In [36]:
a.shape

torch.Size([1, 128, 256])

In [29]:
output.shape,output

(torch.Size([1, 128, 30522]),
 tensor([[[-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726],
          [-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726],
          [-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726],
          ...,
          [-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726],
          [-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726],
          [-0.7168, -0.6143, -1.1625,  ..., -0.2282,  0.1168,  0.6726]]],
        device='cuda:0', grad_fn=<ViewBackward0>))

In [27]:
input_tensor

tensor([[  101,  1045,  2293, 16596,  1011, 10882,  1998,  2572,  5627,  2000,
          2404,  2039,  2007,  1037,  2843,  1012, 16596,  1011, 10882,  5691,
          1013,  2694,  2024,  2788,  2104, 11263, 25848,  1010,  2104,  1011,
         12315,  1998, 28947,  1012,  1045,  2699,  2000,  2066,  2023,  1010,
          1045,  2428,  2106,  1010,  2021,  2009,  2003,  2000,  2204,  2694,
         16596,  1011, 10882,  2004, 17690,  1019,  2003,  2000,  2732, 10313,
          1006,  1996,  2434,  1007,  1012, 10021,  4013,  3367, 20086,  2015,
          1010, 10036, 19747,  4520,  1010, 25931,  3064, 22580,  1010,  1039,
          2290,  2008,  2987,  1005,  1056,  2674,  1996,  4281,  1010,  1998,
         16267,  2028,  1011,  8789,  3494,  3685,  2022,  9462,  2007,  1037,
          1005, 16596,  1011, 10882,  1005,  4292,  1012,  1006,  1045,  1005,
          1049,  2469,  2045,  2024,  2216,  1997,  2017,  2041,  2045,  2040,
          2228, 17690,  1019,  2003,  2204, 16596,  

In [1]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gaurav B V\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate devel

In [ ]:
for epoc in range(0,10):
    epoch_loss = 0
    for i,batch in enumerate(tokenized_datasets):
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        model_output = model(i)
        loss = criterion(outputs.view(-1, config.vocab_size), labels.view(-1))

        # 4. Backward pass (Calculate gradients)
        loss.backward()

        # 5. Update weights
        optimizer.step()

        # Stats
        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch} | Batch: {batch_idx} | Loss: {loss.item():.4f}")
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} complete. Average Loss: {avg_loss:.4f}")

Training loop

In [53]:
raw_input_ids = tokenized_datasets["test"][0]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor = torch.tensor(raw_input_ids).unsqueeze(0).to(device)

# 3. Now run the model
output = model(input_tensor)
print(output.shape)

torch.Size([1, 128, 30522])


In [57]:
raw_input_ids

[101,
 1045,
 2293,
 16596,
 1011,
 10882,
 1998,
 2572,
 5627,
 2000,
 2404,
 2039,
 2007,
 1037,
 2843,
 1012,
 16596,
 1011,
 10882,
 5691,
 1013,
 2694,
 2024,
 2788,
 2104,
 11263,
 25848,
 1010,
 2104,
 1011,
 12315,
 1998,
 28947,
 1012,
 1045,
 2699,
 2000,
 2066,
 2023,
 1010,
 1045,
 2428,
 2106,
 1010,
 2021,
 2009,
 2003,
 2000,
 2204,
 2694,
 16596,
 1011,
 10882,
 2004,
 17690,
 1019,
 2003,
 2000,
 2732,
 10313,
 1006,
 1996,
 2434,
 1007,
 1012,
 10021,
 4013,
 3367,
 20086,
 2015,
 1010,
 10036,
 19747,
 4520,
 1010,
 25931,
 3064,
 22580,
 1010,
 1039,
 2290,
 2008,
 2987,
 1005,
 1056,
 2674,
 1996,
 4281,
 1010,
 1998,
 16267,
 2028,
 1011,
 8789,
 3494,
 3685,
 2022,
 9462,
 2007,
 1037,
 1005,
 16596,
 1011,
 10882,
 1005,
 4292,
 1012,
 1006,
 1045,
 1005,
 1049,
 2469,
 2045,
 2024,
 2216,
 1997,
 2017,
 2041,
 2045,
 2040,
 2228,
 17690,
 1019,
 2003,
 2204,
 16596,
 1011,
 102]

In [58]:
input_tensor,input_tensor.shape

(tensor([[  101,  1045,  2293, 16596,  1011, 10882,  1998,  2572,  5627,  2000,
           2404,  2039,  2007,  1037,  2843,  1012, 16596,  1011, 10882,  5691,
           1013,  2694,  2024,  2788,  2104, 11263, 25848,  1010,  2104,  1011,
          12315,  1998, 28947,  1012,  1045,  2699,  2000,  2066,  2023,  1010,
           1045,  2428,  2106,  1010,  2021,  2009,  2003,  2000,  2204,  2694,
          16596,  1011, 10882,  2004, 17690,  1019,  2003,  2000,  2732, 10313,
           1006,  1996,  2434,  1007,  1012, 10021,  4013,  3367, 20086,  2015,
           1010, 10036, 19747,  4520,  1010, 25931,  3064, 22580,  1010,  1039,
           2290,  2008,  2987,  1005,  1056,  2674,  1996,  4281,  1010,  1998,
          16267,  2028,  1011,  8789,  3494,  3685,  2022,  9462,  2007,  1037,
           1005, 16596,  1011, 10882,  1005,  4292,  1012,  1006,  1045,  1005,
           1049,  2469,  2045,  2024,  2216,  1997,  2017,  2041,  2045,  2040,
           2228, 17690,  1019,  2003,  2

In [49]:
data = torch.randint(1, vocab_size, (1, 128)) 


In [50]:
data,data.shape

(tensor([[29506, 16860,  4107, 18479, 28678, 24782, 25745, 21059, 26154, 14531,
          16816,  7927, 24474, 21221, 29960,  7256, 20306,  8592, 15014, 27298,
           3737,  7998,  5783,  7282, 21600, 13436, 21951,  3138,  6141, 30473,
           2976, 22741,   865,  3218, 26613, 15516, 28411,   580, 19822, 12393,
          26103,  7455, 13860, 23898, 29636,  9260,    80, 21041, 20132, 14894,
          12749, 13159,  4216, 24204, 21542,  4889,  9594, 21358, 25991, 10424,
           3918, 29471, 25386,  6028,  1664, 19653,  4313, 27089,  5414,  4747,
          24610,  1666, 23342, 26522, 10809, 26999, 11571, 24765, 29567, 19796,
          10432,  9883, 18443,  5204, 28166, 12603, 13178, 28980, 22126, 17195,
          17032, 27779, 18415, 30475,   676, 16519, 25320, 21961,  4678, 30459,
           3306, 14498, 28828, 29214, 21324,  1346, 20832,  3161, 11853, 26846,
          20139, 27611, 12246, 21381,   505, 24405,  5297,  2528, 10648,  2394,
           7318, 18244, 27917, 22528, 20

In [65]:
inputs_tensor,inputs_tensor.shape

(tensor([[22181,   283, 15938, 11312, 27779,  6817,  3098, 16158, 26505, 21746,
           8153, 24861, 25239,  9445, 24021,  7538, 21473, 30140,  8188, 21515,
          15653, 11198, 24911,   103,   213, 13981, 14149, 22937,  3371, 17278,
          24886, 20046,  6555,  3818, 21714, 16104, 14705, 24104,  6906,  2375,
          25061, 24118,   394, 15022, 16445,   738, 19001, 10152, 30467, 28367,
           3754,  2998,  6200, 27079,  4156, 16405, 10357,  2829, 13126,  7476,
          27920, 19822,  4526, 30002, 18256, 13474, 22001,  7765, 30119, 11460,
           6456,  9801, 25745, 21924,  3976, 23221,  5167, 22331,  4958,  4116,
          16168,  7718, 18092,  7552, 21854, 18369,  3306, 19485,   633,  8931,
          15698, 20846,  3216,  9437, 20049, 13744,  4421, 23578, 26801,  3762,
           5465,   485, 27439, 16365,   865, 24982,  8445,  4092, 10523, 23324,
            793,  2332, 29706, 27626, 18792, 15618,  1900, 19623, 19011, 22297,
          15728, 14137, 14946, 22344,  2

In [73]:
vocab_size = 30522
mask_token_id = 0  # Let's assume 0 is our [MASK]
data = torch.randint(1, vocab_size, (1, 128)) 
epochs = 100
# 3. The Training Loop
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Create a mask: randomly pick indices to hide (15% chance)
    labels = data.clone()
    labels = labels.to(device)
    probability_matrix = torch.full(labels.shape, 0.15)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    
    # Apply mask to input
    inputs = data.clone()
    inputs_tensor = torch.tensor(inputs).to(device)

    inputs[masked_indices] = mask_token_id 
    
    # Forward pass
    outputs = model(input_tensor)
    # Loss calculation: Only calculate loss on the MASKED tokens
    # We flatten to (batch * seq, vocab) for CrossEntropy
    loss = criterion(outputs[masked_indices], labels[masked_indices])
    
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

C:\Users\Gaurav B V\AppData\Local\Temp\ipykernel_20328\2919630537.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs_tensor = torch.tensor(inputs).to(device)


Epoch 0, Loss: 10.5493
Epoch 20, Loss: 10.1677


KeyboardInterrupt: 